# **Bonus Project:** Classifying NSL-KDD Dataset using DRL

Classify network traffic from the NSL-KDD dataset as normal (0) or anomalous (1) using Deep Reinforcement Learning (DRL). 

A Deep Q-Network (DQN) will be used to train a reinforcement learning agent, which learns to classify traffic based on feedback from its actions.

The model's goal is to optimize its policy and accurately identify normal vs. attack traffic for intrusion detection.


## Exploring Phase 

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('../data/KDD_Shuffled_Combined_Set.csv')

In [4]:
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,http,SF,338,18918,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
1,0,tcp,http,SF,308,4336,0,0,0,0,...,255,1.00,0.00,0.02,0.03,0.00,0.00,0.00,0.00,0.0
2,0,tcp,smtp,SF,1064,338,0,0,0,0,...,198,0.80,0.04,0.01,0.01,0.00,0.00,0.00,0.00,0.0
3,0,tcp,smtp,SF,768,384,0,0,0,0,...,117,0.59,0.03,0.01,0.02,0.01,0.00,0.00,0.00,0.0
4,0,tcp,smtp,S0,0,0,0,0,0,0,...,118,0.39,0.05,0.02,0.02,0.02,0.02,0.01,0.01,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148511,0,tcp,echo,RSTO,0,0,0,0,0,0,...,4,0.02,0.09,0.00,0.00,0.00,0.00,1.00,1.00,1.0
148512,0,tcp,telnet,S0,0,0,0,0,0,0,...,4,0.02,0.07,0.01,0.00,1.00,1.00,0.00,0.00,1.0
148513,0,tcp,smtp,SF,0,83,0,0,0,0,...,124,0.49,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.0
148514,899,tcp,domain,RSTO,1562,0,0,0,0,0,...,2,1.00,0.00,0.50,0.00,0.00,0.00,1.00,1.00,1.0


In [5]:
df.isnull().sum()

duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [6]:
df.dtypes

duration                         int64
protocol_type                   object
service                         object
flag                            object
src_bytes                        int64
dst_bytes                        int64
land                             int64
wrong_fragment                   int64
urgent                           int64
hot                              int64
num_failed_logins                int64
logged_in                        int64
num_compromised                  int64
root_shell                       int64
su_attempted                     int64
num_root                         int64
num_file_creations               int64
num_shells                       int64
num_access_files                 int64
num_outbound_cmds                int64
is_host_login                    int64
is_guest_login                   int64
count                            int64
srv_count                        int64
serror_rate                    float64
srv_serror_rate          

In [7]:
for col_name in ('label', 'protocol_type', 'service', 'flag'):
    unique = df[col_name].unique()
    print(col_name, '\t', len(unique), unique)
    print()

label 	 2 [0. 1.]

protocol_type 	 3 ['tcp' 'udp' 'icmp']

service 	 70 ['http' 'smtp' 'domain_u' 'other' 'courier' 'private' 'ftp_data' 'telnet'
 'eco_i' 'link' 'auth' 'bgp' 'sql_net' 'whois' 'imap4' 'tftp_u' 'gopher'
 'ecr_i' 'netbios_ssn' 'Z39_50' 'X11' 'http_443' 'time' 'ftp' 'uucp_path'
 'pop_3' 'klogin' 'efs' 'discard' 'ssh' 'netbios_dgm' 'supdup' 'uucp'
 'login' 'urp_i' 'kshell' 'echo' 'finger' 'hostnames' 'netstat' 'domain'
 'nnsp' 'ldap' 'iso_tsap' 'ctf' 'pop_2' 'nntp' 'exec' 'daytime' 'vmnet'
 'csnet_ns' 'netbios_ns' 'systat' 'mtp' 'ntp_u' 'remote_job' 'name' 'IRC'
 'rje' 'urh_i' 'sunrpc' 'printer' 'red_i' 'shell' 'tim_i' 'aol' 'pm_dump'
 'harvest' 'http_2784' 'http_8001']

flag 	 11 ['SF' 'S0' 'REJ' 'RSTR' 'RSTO' 'S1' 'SH' 'S2' 'S3' 'OTH' 'RSTOS0']



## Preprocessing Phase

In [8]:
col_names = ['protocol_type', 'service', 'flag']

df_encoded = pd.get_dummies(df, columns=col_names)
df_encoded

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,338,18918,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
1,0,308,4336,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
2,0,1064,338,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
3,0,768,384,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
4,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148511,0,0,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
148512,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
148513,0,0,83,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
148514,899,1562,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False


In [9]:
df_encoded.dtypes

duration          int64
src_bytes         int64
dst_bytes         int64
land              int64
wrong_fragment    int64
                  ...  
flag_S1            bool
flag_S2            bool
flag_S3            bool
flag_SF            bool
flag_SH            bool
Length: 123, dtype: object

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns)
df_scaled['label'] = df_encoded['label']
df_scaled['label']

0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
148511    1.0
148512    1.0
148513    0.0
148514    1.0
148515    0.0
Name: label, Length: 148516, dtype: float64

In [11]:
from sklearn.model_selection import train_test_split

X = df_scaled.drop('label', axis=1)
y = df_scaled['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Creating Environment

In [12]:
import gymnasium as gym       # Modern OpenAI Gym
from gymnasium import spaces
import random

class NSLKDDEnv(gym.Env):
    def __init__(self, X, y):
        super(NSLKDDEnv, self).__init__()

        # Attached dataset can be test or train depending on the phase
        self.X = X.reset_index(drop=True)
        self.y = y.reset_index(drop=True)

        # Initial State
        self.reset()
        
        # Actions are Normal and Anomaly
        self.action_space = spaces.Discrete(2)
        
        # Space is simply the features
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.X.shape[1],), dtype=np.float32)

    def reset(self):
        # Randomly select any row from the linked dataset
        self.loc = 0
        self.state = self.X.loc[self.loc]
        return self.state

    def step(self, action):
        # Postive reward if correct action
        # Negative reward if wrong action
        reward = 1 if action == self.y.loc[self.loc] else -1

        # Next location
        self.loc += 1

        # Done if there is no more remaining locations
        done = True
        if self.loc < self.X.shape[0]:
            self.state = self.X.loc[self.loc]
            done = False
            
        return self.state, reward, done

    def render(self):
        print(f'NSLKDDEnv: location {self.loc}')

In [13]:
env = NSLKDDEnv(X_train, y_train)

state = env.reset()
done = False
while not done:
    action = env.action_space.sample()  # Random action (select feature index)
    new_state, reward, done = env.step(action)
    print(f'Loc {env.loc}, Action {action}, Reward {reward}')
    state = new_state
    if env.loc == 10:
        break

Loc 1, Action 1, Reward -1
Loc 2, Action 1, Reward 1
Loc 3, Action 1, Reward 1
Loc 4, Action 1, Reward 1
Loc 5, Action 0, Reward 1
Loc 6, Action 0, Reward 1
Loc 7, Action 0, Reward 1
Loc 8, Action 0, Reward -1
Loc 9, Action 0, Reward -1
Loc 10, Action 0, Reward -1


## Training Phase, DRL Model

In [14]:
import tensorflow as tf
from tensorflow.keras import models, layers, optimizers

In [15]:
input_shape = env.observation_space.shape[0]
num_actions = env.action_space.n
input_shape, num_actions

(122, np.int64(2))

In [ ]:

q_model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_shape,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(num_actions, activation='linear')  # Outputs Q-values for each action
])

In [ ]:

q_model.summary()